# lltAttention, Causal Self-Attention, Built and Tested

LLM Training Lab, module 3. The cells below are the lab page's code, in the same order. Steps 1 to 3 need only NumPy. Steps 4 to 6 and 8 run on a CPU. Step 7 needs a CUDA GPU, such as a free Colab T4.

Recorded results for every step: `../../runs/lltAttention/`.

In [ ]:
# Runtime check. Run this first. It prints what to fix before the lab cells run.
import platform, shutil, sys
ok = True
if sys.version_info < (3, 11):
    ok = False
    print("Python", platform.python_version(), "is too old. Use 3.11 or later (Colab: Runtime > Change runtime type).")
try:
    import torch, numpy
    print("python", platform.python_version(), "| torch", torch.__version__, "| numpy", numpy.__version__)
except ImportError as e:
    ok = False
    print("Missing package:", e.name, "| install it with: pip install torch==2.15.0 numpy")
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f"RAM {ram:.1f} GB")
    if ram < 6:
        print("Step 6 needs about 3 GB free at T = 4,096. Close other programs or lower the largest T.")
except ImportError:
    pass
free = shutil.disk_usage(".").free / 1e9
print(f"free disk {free:.1f} GB")
if free < 1:
    ok = False
    print("Free at least 1 GB of disk for the 22 MB TinyStories download in step 8.")
gpu = "torch" in globals() and torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if gpu else "none. Steps 1 to 6 and 8 run on the CPU. Step 7 needs a GPU "
      "(Colab: Runtime > Change runtime type > T4 GPU).")
print("Runtime OK" if ok else "Fix the lines above, then run this cell again.")

## Step 1. Write single-head causal attention in NumPy

In [ ]:
import numpy as np

def softmax(s):
    s = s - s.max(axis=-1, keepdims=True)   # the largest score in each row becomes 0
    e = np.exp(s)
    return e / e.sum(axis=-1, keepdims=True)

def causal_attention(x, Wq, Wk, Wv):
    q, k, v = x @ Wq, x @ Wk, x @ Wv                     # each (T, d)
    T, d = q.shape
    scores = q @ k.T / np.sqrt(d)                        # (T, T), row i = query i
    future = np.triu(np.ones((T, T), dtype=bool), k=1)   # True where key j > query i
    scores = np.where(future, -np.inf, scores)           # mask BEFORE the softmax
    w = softmax(scores)                                  # (T, T), each row sums to 1
    return w @ v, w                                      # (T, d) and (T, T)

rng = np.random.default_rng(0)
T, C, d = 6, 16, 8          # tokens, model width, head width
x = rng.standard_normal((T, C))
Wq, Wk, Wv = (rng.standard_normal((C, d)) / np.sqrt(C) for _ in range(3))

y, w = causal_attention(x, Wq, Wk, Wv)
print("x", x.shape, "| q, k, v", (x @ Wq).shape, "| weights", w.shape, "| y", y.shape)
print("row sums", np.round(w.sum(axis=1), 6))
print("weight on future tokens", w[np.triu(np.ones((T, T), dtype=bool), k=1)].sum())
print(np.round(w, 2))

## Step 2. Split the width into heads

In [ ]:
import numpy as np

def softmax(s):
    s = s - s.max(axis=-1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=-1, keepdims=True)

def split_heads(a, h):          # (B, T, C) -> (B, h, T, d), d = C // h
    B, T, C = a.shape
    return a.reshape(B, T, h, C // h).transpose(0, 2, 1, 3)

def merge_heads(a):             # (B, h, T, d) -> (B, T, h * d)
    B, h, T, d = a.shape
    return a.transpose(0, 2, 1, 3).reshape(B, T, h * d)

def attend(q, k, v):            # (B, h, T, d) each -> (B, h, T, d)
    T, d = q.shape[-2:]
    scores = q @ k.swapaxes(-1, -2) / np.sqrt(d)               # (B, h, T, T)
    scores = np.where(np.triu(np.ones((T, T), dtype=bool), k=1), -np.inf, scores)
    return softmax(scores) @ v

def mha(x, Wqkv, Wo, h, split=split_heads):
    q, k, v = np.split(x @ Wqkv, 3, axis=-1)                    # each (B, T, C)
    return merge_heads(attend(split(q, h), split(k, h), split(v, h))) @ Wo

rng = np.random.default_rng(0)
B, T, C, h = 2, 6, 16, 4
d = C // h
x = rng.standard_normal((B, T, C))
Wqkv = rng.standard_normal((C, 3 * C)) / np.sqrt(C)
Wo = rng.standard_normal((C, C)) / np.sqrt(C)
y = mha(x, Wqkv, Wo, h)

# Reference: each head on its own slice of d columns, one head at a time
q, k, v = np.split(x @ Wqkv, 3, axis=-1)
heads = [attend(q[:, None, :, i * d:(i + 1) * d], k[:, None, :, i * d:(i + 1) * d],
                v[:, None, :, i * d:(i + 1) * d])[:, 0] for i in range(h)]
ref = np.concatenate(heads, axis=-1) @ Wo
print("x", x.shape, "| one head of q", split_heads(q, h).shape, "| y", y.shape)
print("max abs difference from the per-head loop", np.abs(y - ref).max())

# A common bug: reshape straight to (B, h, T, d) with no transpose
wrong = lambda a, h: a.reshape(a.shape[0], h, a.shape[1], -1)
print("same shape?", mha(x, Wqkv, Wo, h, wrong).shape == y.shape,
      "| max abs difference", round(float(np.abs(mha(x, Wqkv, Wo, h, wrong) - ref).max()), 3))

## Step 3. Check the 1/sqrt(d) scale with numbers

In [ ]:
import numpy as np

def softmax(s):
    s = s - s.max(axis=-1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=-1, keepdims=True)

def grad_norm(p):
    # Size of the softmax gradient, the Frobenius norm of diag(p) - p p^T, per row
    sq = (p ** 2).sum(-1)
    return np.sqrt(np.maximum(sq - 2 * (p ** 3).sum(-1) + sq ** 2, 0))

rng = np.random.default_rng(0)
n, T = 4_000, 64            # n query-key pairs for the variance, T keys per query
print("     d   var(q.k)   var / d")
for d in [16, 64, 256, 1024]:
    q, k = rng.standard_normal((n, d)), rng.standard_normal((n, d))
    dots = (q * k).sum(axis=1)                  # n dot products of unit-variance vectors
    print(f"{d:6d} {dots.var():10.1f} {dots.var() / d:9.3f}")

print()
print("     d   top weight        rows above 0.99   median gradient")
print("         raw    scaled     raw    scaled     raw      scaled")
for d in [16, 64, 256, 1024]:
    Q, K = rng.standard_normal((500, d)), rng.standard_normal((T, d))
    raw = Q @ K.T                               # 500 queries, each scored against T keys
    pr, ps = softmax(raw), softmax(raw / np.sqrt(d))
    print(f"{d:6d}   {pr.max(-1).mean():.3f}  {ps.max(-1).mean():.3f}     "
          f"{(pr.max(-1) > 0.99).mean():.3f}  {(ps.max(-1) > 0.99).mean():.3f}     "
          f"{np.median(grad_norm(pr)):.5f}  {np.median(grad_norm(ps)):.3f}")

## Step 4. Build the PyTorch module and match SDPA

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, C, h, dropout=0.0):
        super().__init__()
        assert C % h == 0, "the width C must split evenly into h heads"
        self.h, self.d, self.p = h, C // h, dropout
        self.qkv = nn.Linear(C, 3 * C, bias=False)
        self.proj = nn.Linear(C, C, bias=False)

    def attend(self, q, k, v):                                    # (B, h, T, d) each
        T = q.shape[-2]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d)      # (B, h, T, T)
        future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=q.device), 1)
        w = torch.softmax(scores.masked_fill(future, float("-inf")), dim=-1)
        return F.dropout(w, self.p, training=self.training) @ v   # (B, h, T, d)

    def forward(self, x):                                         # (B, T, C)
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)                    # (B, T, C) each
        q, k, v = (a.view(B, T, self.h, self.d).transpose(1, 2) for a in (q, k, v))
        y = self.attend(q, k, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class SDPAAttention(CausalSelfAttention):                         # same weights, PyTorch's kernel
    def attend(self, q, k, v):
        p = self.p if self.training else 0.0
        return F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=True)

torch.manual_seed(0)
torch.set_num_threads(4)
B, T, C, h = 4, 128, 256, 8
x = torch.randn(B, T, C)
for dtype, tol in [(torch.float32, 1e-5), (torch.float64, 1e-12)]:
    ours = CausalSelfAttention(C, h).to(dtype).eval()
    ref = SDPAAttention(C, h).to(dtype).eval()
    ref.load_state_dict(ours.state_dict())
    with torch.no_grad():
        a, b = ours(x.to(dtype)), ref(x.to(dtype))
    print(str(dtype).ljust(14), "max abs error", f"{(a - b).abs().max().item():.2e}",
          "| allclose", torch.allclose(a, b, rtol=0, atol=tol))
print("parameters", sum(p.numel() for p in ours.parameters()), "= 4 x C x C =", 4 * C * C)

## Step 5. Prove the layer cannot see ahead

In [ ]:
# Uses CausalSelfAttention, SDPAAttention and C from step 4. Run this cell after it.
def leakage(attn, T=32, seed=0):
    # For each t, change token t + 1 and measure how far outputs at positions 0..t move.
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(1, T, C, generator=g)
    worst = 0.0
    with torch.no_grad():
        y = attn(x)
        for t in range(T - 1):
            x2 = x.clone()
            x2[:, t + 1] += torch.randn(C, generator=g)
            worst = max(worst, (y[:, : t + 1] - attn(x2)[:, : t + 1]).abs().max().item())
    return worst

class NoMask(CausalSelfAttention):
    def attend(self, q, k, v):
        return torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.d), dim=-1) @ v

class MaskAfterSoftmax(CausalSelfAttention):
    def attend(self, q, k, v):
        T = q.shape[-2]
        w = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.d), dim=-1)
        future = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        return w.masked_fill(future, 0.0) @ v                   # zeroed after normalizing

torch.manual_seed(0)
base = CausalSelfAttention(C, h).eval()
for cls in [CausalSelfAttention, SDPAAttention, NoMask, MaskAfterSoftmax]:
    attn = cls(C, h).eval()
    attn.load_state_dict(base.state_dict())
    worst = leakage(attn)
    print(cls.__name__.ljust(20), "largest change at positions <= t:", f"{worst:.2e}",
          "PASS" if worst < 1e-6 else "LEAKS")

## Step 6. Measure attention memory on the CPU

In [ ]:
# Peak memory of one causal attention call on the CPU, naive vs PyTorch SDPA.
from torch.profiler import profile, ProfilerActivity

def naive(q, k, v):
    T = q.shape[-2]
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])     # (B, h, T, T)
    future = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
    return torch.softmax(scores.masked_fill(future, float("-inf")), dim=-1) @ v

def sdpa(q, k, v):
    return F.scaled_dot_product_attention(q, k, v, is_causal=True)

def peak_mib(fn, *args):
    # Replay every allocation and free the profiler recorded, in time order, and keep the peak.
    with profile(activities=[ProfilerActivity.CPU], profile_memory=True) as prof:
        fn(*args)
    events = sorted((e.start_ns(), e.nbytes()) for e in prof.profiler.kineto_results.events()
                    if e.name() == "[memory]")
    live = peak = 0
    for _, nbytes in events:
        live += nbytes
        peak = max(peak, live)
    return peak / 2**20

B, h, d = 1, 12, 64                      # one sequence, GPT-2 small's 12 heads of width 64
memory = {}
print("     T   T x T scores   naive peak   SDPA peak   naive / SDPA   (MiB)")
for T in [128, 256, 512, 1024, 2048, 4096]:
    q, k, v = (torch.randn(B, h, T, d) for _ in range(3))
    scores = B * h * T * T * 4 / 2**20     # float32 is 4 bytes
    n, s = peak_mib(naive, q, k, v), peak_mib(sdpa, q, k, v)
    memory[T] = (scores, n, s)
    print(f"{T:6d} {scores:13.1f} {n:12.1f} {s:11.1f} {n / s:13.1f}")

## Step 7. Measure naive attention and SDPA on a T4

Skip this cell on a CPU runtime. The table on the lab page is our run on a Kaggle T4.

In [ ]:
# Needs a CUDA GPU (Colab or Kaggle T4). Naive causal attention vs SDPA, float16, batch 4, 12 heads of width 64.
import math, statistics
import torch
import torch.nn.functional as F

def naive(q, k, v):
    T = q.shape[-2]
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=q.device), 1)
    return torch.softmax(scores.masked_fill(future, float("-inf")), dim=-1) @ v

def sdpa(q, k, v):
    return F.scaled_dot_product_attention(q, k, v, is_causal=True)

def measure(fn, T, backward, B=4, h=12, d=64, iters=20):
    q, k, v = (torch.randn(B, h, T, d, device="cuda", dtype=torch.float16, requires_grad=backward) for _ in range(3))
    g = torch.randn_like(q)                     # the gradient arriving from the layer above
    def call():
        y = fn(q, k, v)
        if backward:
            y.backward(g)
            q.grad = k.grad = v.grad = None     # count the gradients inside every call
    for _ in range(3):
        call()                                  # warm-up
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    call()
    torch.cuda.synchronize()
    peak = (torch.cuda.max_memory_allocated() - base) / 2**20
    times = []
    for _ in range(iters):
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        call()
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))
    return peak, statistics.median(times)

print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0), torch.__version__)
for backward in (False, True):
    print("forward + backward" if backward else "forward only")
    print("     T   naive MiB   SDPA MiB   naive ms   SDPA ms")
    for T in [128, 256, 512, 1024, 2048, 4096]:
        (nm, nt), (sm, st) = measure(naive, T, backward), measure(sdpa, T, backward)
        print(f"{T:6d} {nm:11.1f} {sm:10.1f} {nt:10.2f} {st:9.2f}")

## Step 8. Train a small model with a broken mask

Uses the classes from steps 4 and 5.

In [ ]:
# Train the same small character model with three attention classes from steps 4 and 5.
import urllib.request

URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt"
text = urllib.request.urlopen(URL).read().decode("utf-8")[:1_000_000]
chars = sorted(set(text))
data = torch.tensor([chars.index(c) for c in text])
train, val = data[:900_000], data[900_000:]

class Block(nn.Module):
    def __init__(self, attn_cls, C, h):
        super().__init__()
        self.n1, self.n2 = nn.RMSNorm(C), nn.RMSNorm(C)
        self.attn = attn_cls(C, h)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))

class TinyLM(nn.Module):
    def __init__(self, attn_cls, V, C=64, h=4, L=4, ctx=128):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, C), nn.Embedding(ctx, C)
        self.blocks = nn.Sequential(*[Block(attn_cls, C, h) for _ in range(L)])
        self.head = nn.Linear(C, V)

    def forward(self, idx):
        return self.head(self.blocks(self.tok(idx) + self.pos(torch.arange(idx.shape[1]))))

def batch(split, B=32, T=128, g=None):
    i = torch.randint(len(split) - T - 1, (B,), generator=g)
    return torch.stack([split[j:j + T] for j in i]), torch.stack([split[j + 1:j + T + 1] for j in i])

@torch.no_grad()
def prefix_loss(model, x, y):     # score position t from tokens 0..t only, as generation does
    return torch.stack([F.cross_entropy(model(x[:, :t + 1])[:, -1], y[:, t]) for t in range(x.shape[1])]).mean().item()

xv, yv = batch(val, B=64, g=torch.Generator().manual_seed(1))
results, curves, models = {}, {}, {}
for cls in [CausalSelfAttention, NoMask, MaskAfterSoftmax]:
    torch.manual_seed(0)
    model = TinyLM(cls, len(chars))
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    g = torch.Generator().manual_seed(0)
    curve = []
    for step in range(600):
        x, y = batch(train, g=g)
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad()
        loss.backward()
        opt.step()
        curve.append(loss.item())
    model.eval()
    with torch.no_grad():
        full = F.cross_entropy(model(xv).flatten(0, 1), yv.flatten()).item()
    results[cls.__name__] = (loss.item(), full, prefix_loss(model, xv, yv))
    curves[cls.__name__], models[cls.__name__] = curve, model
    print(cls.__name__.ljust(20), "train %.3f | val, full sequence %.3f | val, prefix only %.3f" % results[cls.__name__])

## Prove it

Fill in the four functions, then run the cell. It prints `All checks passed` when they are right.

In [ ]:
import numpy as np

def split_heads(a, h):
    # TODO: (B, T, C) -> (B, h, T, C // h). Head i must hold columns i*d to (i+1)*d - 1.
    return None

def merge_heads(a):
    # TODO: (B, h, T, d) -> (B, T, h * d), the exact inverse of split_heads
    return None

def causal_mask(T):
    # TODO: a (T, T) boolean array, True where key j is in the future of query i (j > i)
    return None

def stable_softmax(s):
    # TODO: softmax over the last axis that works for scores like 1000 and for -inf
    return None

def attention(x, Wqkv, Wo, h):
    q, k, v = (split_heads(a, h) for a in np.split(x @ Wqkv, 3, axis=-1))
    T, d = q.shape[-2:]
    scores = np.where(causal_mask(T), -np.inf, q @ k.swapaxes(-1, -2) / np.sqrt(d))
    return merge_heads(stable_softmax(scores) @ v) @ Wo

# Checks. Run the cell. It prints All checks passed when all four functions are right.
rng = np.random.default_rng(0)
B, T, C, h = 2, 6, 16, 4
x = rng.standard_normal((B, T, C))
assert split_heads(x, h) is not None, "split_heads returns None"
assert split_heads(x, h).shape == (2, 4, 6, 4), "split_heads should give (B, h, T, d)"
assert np.array_equal(split_heads(x, h)[:, 1], x[:, :, 4:8]), "head 1 should hold columns 4 to 7"
assert np.array_equal(merge_heads(split_heads(x, h)), x), "merge_heads(split_heads(x)) should give x back"
assert causal_mask(3) is not None, "causal_mask returns None"
assert causal_mask(3).tolist() == [[False, True, True], [False, False, True], [False, False, False]], "mask is wrong"
p = stable_softmax(np.array([[1000.0, 1001.0, 1002.0], [0.0, -np.inf, 1.0]]))
assert p is not None and not np.isnan(p).any(), "softmax gives NaN, subtract the row max first"
assert np.allclose(p[0], [0.09003057, 0.24472847, 0.66524096]), "softmax values are wrong"
assert p[1, 1] == 0.0 and np.allclose(p.sum(-1), 1.0), "-inf should get weight 0 and rows should sum to 1"
Wqkv, Wo = rng.standard_normal((C, 3 * C)) / 4, rng.standard_normal((C, C)) / 4
y = attention(x, Wqkv, Wo, h)
for t in range(T - 1):
    x2 = x.copy()
    x2[:, t + 1] += 1.0
    moved = np.abs(attention(x2, Wqkv, Wo, h)[:, : t + 1] - y[:, : t + 1]).max()
    assert moved == 0.0, f"changing token {t + 1} moved the output at positions <= {t}"
print("y", y.shape, "| leakage test passed for t = 0 to", T - 2)
print("All checks passed")